# Проекты в рамках курса

В течение курса будут опубликованы четыре проектных задания:

1. Point Cloud.
2. Работа с текстами.
3. Рекомендательные системы.
4. VLM.

Набор обязательных работ зависит от выбранного уровня домашних заданий.

**Усложненные домашние задания.** Проект по VLM засчитывается как последнее усложненное домашнее задание. Дополнительно нужно выполнить два проекта на выбор из первых трех.

**Базовые домашние задания.** Нужно выполнить два проекта: проект по VLM и один проект на выбор из первых трех.

Проекты будут публиковаться по ходу курса. Все выбранные проекты нужно сдать до 13 декабря 2026 года, 22:00.


# Творческий проект 1. Point Cloud

Работа индивидуальная. Выберите один вариант проекта и выполните все пять заданий для него. Каждый блок оценивается в 20 баллов, максимум за проект - **100 баллов**. Выполнять оба варианта не требуется.

Работа выполняется во внутренней платформе Culab. Заранее проверьте, что вам доступен GPU, и выполните тестовый запуск модели. Если GPU недоступен, сообщите команде курса.

Можно использовать готовую реализацию PointNet и фрагменты семинара, а также адаптировать их для проекта. Укажите происхождение заимствованных частей и поясните, как они работают. Нужно получить рабочий baseline, предложить и проверить изменения, добиться понятного улучшения относительно базовой модели и разобрать результаты.

Число запусков с разными seed студент выбирает самостоятельно. Несколько seed не являются обязательным условием полного балла. Зафиксируйте использованные seed и укажите, на каких запусках основаны выводы.


## Материалы для старта

Данные:

- [ModelNet40: официальный набор в формате OFF](https://modelnet.cs.princeton.edu/download.html)
- [S3DIS: страница официального набора](http://buildingparser.stanford.edu/dataset.html). Для проекта нужна версия `Stanford3dDataset_v1.2_Aligned_Version.zip`.

Реализации, которые можно использовать как основу:

- [PointNet и PointNet++ на PyTorch](https://github.com/yanx27/Pointnet_Pointnet2_pytorch)
- [Pointcept](https://github.com/Pointcept/Pointcept)

При использовании стороннего кода укажите источник и отделите заимствованные части от собственных изменений.


## Варианты проекта

| | ModelNet40: классификация | S3DIS: сегментация |
|---|---|---|
| Данные | Вершины CAD-моделей из файлов OFF | S3DIS aligned version, точки XYZRGB |
| Вход и выход | Вход $(B,N,3)$, выход $(B,40)$, одна метка на объект | Вход $(B,N,C)$, выход $(B,N,13)$, одна метка на точку |
| Метрики | Accuracy и mean class accuracy. Основную метрику выбора модели укажите до сравнения конфигураций | Основная метрика - mIoU. Дополнительно: IoU по классам и point accuracy |
| Разбиение | Официальный train разделите на внутренние train и val. Официальный test оставьте для финальной оценки | Test - Area 5. Val выделите по целым комнатам или используйте Area 6; остальные разрешенные области - train |

В S3DIS потребуется нарезка блоков и сборка предсказаний целой комнаты. Способ покрытия комнаты и объединения предсказаний выберите самостоятельно. Каждая исходная точка должна получить ответ и учитываться в итоговых метриках ровно один раз.


## Распределение баллов

| Задание | Содержание | Баллы |
|---|---|---:|
| 1 | Данные и разбиение | 20 |
| 2 | Базовая модель и первичные проверки | 20 |
| 3 | Обучение и воспроизводимость | 20 |
| 4 | Сравнимые эксперименты | 20 |
| 5 | Улучшение, финальная оценка и анализ | 20 |
| | **Итого** | **100** |


## Задание 1. Данные и разбиение - 20 баллов

Подготовьте `Dataset` и `DataLoader`. Зафиксируйте список классов, разбиение и предобработку. Покажите несколько примеров до и после преобразований, формы батча, типы и диапазоны значений.

**ModelNet40.** Прочитайте координаты вершин OFF, выберите число точек $N$, реализуйте выборку с обработкой случая, когда точек меньше $N$. Центрируйте облако и нормируйте масштаб. Сделайте внутренний train/val split по объектам с сохранением классов в обеих частях.

**S3DIS.** Соберите точки и метки комнат из аннотаций, зафиксируйте соответствие 13 классов. Подготовьте блоки и выборку точек, приведите RGB к $[0,1]$, опишите используемые координаты. Индексы выборки и перестановки должны совпадать для точек и меток. Блоки одной комнаты не могут попасть в разные части.

### За что можем снижать баллы

- Разбиение не зафиксировано или есть пересечения между train, val и test.
- Есть ошибки чтения, выборки или нормализации данных; в S3DIS точки и метки рассогласованы.
- `Dataset` и `DataLoader` возвращают неверные формы, типы или индексы классов.
- Нет численных проверок и визуализаций, позволяющих проверить предобработку.


## Задание 2. Базовая модель и первичные проверки - 20 баллов

Обучаемый baseline должен соответствовать задаче. В ModelNet40 достаточно общей MLP по точкам, симметричного pooling и классификатора по глобальному признаку. В S3DIS объедините признак каждой точки с глобальным признаком блока и предсказывайте ее класс. T-Net и PointNet++ не обязательны.

Покажите архитектуру, число параметров и forward одного батча. Проверьте кросс-энтропию и градиенты. Без аугментаций переобучите модель на небольшом фиксированном наборе и покажите кривые. Зафиксируйте базовую конфигурацию до экспериментов.

### За что можем снижать баллы

- Архитектура или агрегация признаков не соответствует задаче, выход модели имеет неверную форму.
- Нет описания модели и числа параметров, неверно рассчитан loss или не проверены градиенты.
- Нет проверки переобучения на малом наборе и кривых; модель не запоминает набор, а причина не разобрана.


## Задание 3. Обучение и воспроизводимость - 20 баллов

Обучите baseline на выбранном train. Разделяйте режимы train и eval, отключайте обучающие аугментации при оценке. По эпохам сохраняйте loss, train/val-метрики и learning rate. Лучшую эпоху выбирайте по заранее указанной val-метрике.

Сохраните конфигурацию, seed, split, соответствие классов и веса лучшей модели. Проверьте загрузку весов и повторную оценку. В S3DIS validation проводится на целых комнатах: учитывайте все исходные точки, накапливайте общую матрицу ошибок и затем вычисляйте mIoU.

### За что можем снижать баллы

- Обучение реализовано с ошибками: параметры не обновляются, перепутаны train/eval или при оценке используются обучающие аугментации.
- Метрики рассчитаны неверно, checkpoint выбран не по val или есть утечка данных; в S3DIS нет оценки целых комнат.
- Нет логов, сохраненной конфигурации или рабочего checkpoint, не показана повторная оценка после загрузки весов.


## Задание 4. Сравнимые эксперименты - 20 баллов

Проведите **минимум три сравнения**. В каждом меняется один исследуемый фактор, остальные условия сохраняются. Baseline можно использовать в нескольких сравнениях. Повтор той же конфигурации с другим seed не заменяет исследование нового изменения.

Можно исследовать признаки, аугментации, регуляризацию, число точек или архитектуру. Для ModelNet40 подходят, например, сравнения с аугментациями и без них. Для S3DIS - XYZ против XYZRGB или обучение с весами классов и без них. Варианты выбираете самостоятельно.

Для каждого сравнения запишите ожидание, изменяемый фактор и фиксированные условия. В общей таблице приведите конфигурации, настройки обучения и val-метрики. Если изменение не помогло, это тоже результат: объясните наблюдение и не подменяйте его ожидаемым выводом.

### За что можем снижать баллы

- Выполнено меньше трех содержательных сравнений или результаты запусков нельзя проверить.
- Сравнения несопоставимы: меняется val, одновременно меняются необоснованные факторы или используется test для подбора.
- Нет общей таблицы с конфигурациями, настройками обучения и val-метриками.
- Выводы отсутствуют, противоречат результатам или не объясняют исследуемое изменение.


## Задание 5. Улучшение, финальная оценка и анализ - 20 баллов

Выберите итоговую конфигурацию по val и покажите ее улучшение относительно рабочего baseline. Сравнение проводится на одном split по одной и той же основной метрике, указанной до экспериментов. Не ослабляйте baseline искусственно: у него должны быть корректные данные, обучение и выбор checkpoint по val.

Для ModelNet40 сравнивается выбранная основная метрика: accuracy или mean class accuracy. Для S3DIS - mIoU. Укажите абсолютную разницу значений, число использованных seed и ограничения вывода. Изменение должно быть объяснено и подтверждено сопоставимыми результатами.

После фиксации настроек оцените итоговую модель на официальном test ModelNet40 или на всей Area 5 S3DIS. По test не меняйте настройки и не выбирайте модель. Приведите метрики по классам и разберите успешные и ошибочные предсказания. В S3DIS покажите RGB, разметку и предсказания нескольких целых комнат.

### За что можем снижать баллы

- Нет подтвержденного улучшения рабочего baseline, сравнение некорректно или выбранное изменение не объяснено.
- Нет финальной оценки, оценен неполный test или настройки подбирались по test.
- Нет метрик по классам, визуализаций предсказаний и содержательного разбора ошибок.
- Комплект сдачи неполный, результаты или инструкции не позволяют воспроизвести работу.


## Как качество влияет на оценку

Абсолютных порогов accuracy или mIoU нет. Корректно выполненные и разобранные сравнения могут получить полный балл за задание 4, даже если отдельные изменения ухудшили результат. Полный балл за задание 5 предполагает подтвержденное улучшение относительно baseline. Если улучшения нет, оценка этого блока снижается; корректная финальная оценка и анализ ошибок при этом учитываются.

## Общие правила

Каждое задание оценивается целиком от 0 до 20 баллов. Полный балл ставится за корректное выполнение с показанными результатами. При неполном выполнении учитываются значимость ошибки и ее влияние на результат. Одну и ту же ошибку внутри задания повторно не штрафуем. Системная ошибка может повлиять на несколько заданий: например, утечка делает некорректными split, выбор модели и оценку улучшения.

Если задание отсутствует или его результат нельзя проверить, за него ставится ноль. Если ноутбук не запускается до конца, оцениваются части, которые можно проверить. За готовую реализацию PointNet или код семинара баллы не снижаются при указании происхождения и корректном использовании.


## Что сдавать

Ссылку на ноутбук в Culab и файл `.ipynb` с кодом, результатами и отчетом. Внешние модули приложите вместе с инструкцией запуска.

В комплекте должны быть:

- конфигурации baseline и итоговой модели;
- веса для повторной оценки;
- кривые обучения;
- таблица сравнений;
- итоговые метрики;
- визуализации;
- выводы по экспериментам.

Работа должна показывать ваш вклад и позволять воспроизвести полученные результаты.
